# SQL Essentials Notebook

This notebook covers essential SQL topics with practical examples using the [Chinook sample database](https://github.com/lerocha/chinook-database). We will work through the following topics:

- Introduction to SQL & Setting Up the Database
- Exploring the Database Schema
- Basic Queries: Filtering, Sorting, and Conditional Statements
- Aggregate Functions and Group Summaries
- Combining Tables: Joins & Set Operators
- Querying SQL from Python
- SQL Subqueries: Scalar, Multi-row, Nested, and Correlated
- Common Table Expressions (CTEs) and Views
- Window Functions

---

## Table of Contents

1. [Introduction & Setup](#introduction--setup)
2. [Exploring the Database Schema](#exploring-the-database-schema)
3. [Basic SQL Queries](#basic-sql-queries)
   - [Filtering Data](#filtering-data)
   - [Sorting Results](#sorting-results)
   - [Conditional Statements](#conditional-statements)
4. [Aggregate Functions & Group Summaries](#aggregate-functions--group-summaries)
5. [Combining Tables: Joins & Set Operators](#combining-tables-joins--set-operators)
6. [Querying SQL from Python](#querying-sql-from-python)
7. [SQL Subqueries](#sql-subqueries)
8. [Common Table Expressions (CTE) & Views](#common-table-expressions-cte--views)
9. [Window Functions](#window-functions)
10. [Conclusion](#conclusion)

---

## 1. Introduction & Setup

In this section we introduce SQL and set up our environment.

### 1.1 What is SQL?

SQL (Structured Query Language) is a standard language for interacting with relational databases. It allows you to create, read, update, and delete data in a structured way.

### 1.2 Setting Up the Database

We are using the **Chinook SQLite database**. If you haven’t already, download it from [here](https://github.com/lerocha/chinook-database) and place the `Chinook_Sqlite.sqlite` file in your working directory.

![Data Model](../images/sql_db.png)


There are two ways to connect to the database:

#### Option A: Using Python's `sqlite3` module


In [1]:
# Import Libraries
import sqlite3
import pandas as pd

In [2]:
# Connect to the Chinook SQLite database
conn = sqlite3.connect('../data/Chinook_Sqlite.sqlite')
cursor = conn.cursor()

In [3]:
# Test connection by listing tables
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print("Tables in the database:", tables)

Tables in the database: [('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]


#### Option B: Using the `ipython-sql` Extension

In [4]:
%load_ext sql
# Connect to the database (using the file-based connection string)
%sql sqlite:///../data/Chinook_Sqlite.sqlite



## 2 Exploring the Database Schema
Before querying data, it’s important to know the structure of your database.

### 2.1 List All Tables
Using the ipython-sql magic:

There are two ways to connect to the database:

#### Using the ipython-sql magic:

In [5]:
%%sql
SELECT name 
FROM sqlite_master 
WHERE type='table'
ORDER BY name;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


name
Album
Artist
Customer
Employee
Genre
Invoice
InvoiceLine
MediaType
Playlist
PlaylistTrack


#### Or with Python:

In [6]:
cursor.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
tables = cursor.fetchall()
print("Tables:", tables)


Tables: [('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]


## 3 Basic SQL Queries
Let’s start with some simple SELECT queries.

### Filtering Data
### 3.1 Filtering with Numbers
For example, list invoices with a total greater than 10:

In [7]:
%%sql
SELECT InvoiceId, Total
FROM Invoice
WHERE Total > 10
LIMIT 5;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


InvoiceId,Total
5,13.86
12,13.86
19,13.86
26,13.86
33,13.86


### 3.2 Filtering with Strings/Categories
List artists whose names start with "A":

In [8]:
%%sql
SELECT ArtistId, Name
FROM Artist
WHERE Name LIKE 'A%'
LIMIT 5;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


ArtistId,Name
1,AC/DC
2,Accept
3,Aerosmith
4,Alanis Morissette
5,Alice In Chains


### Sorting Results
Sort the list of invoices by Total in descending order:

In [9]:
%%sql
SELECT InvoiceId, Total
FROM Invoice
ORDER BY Total DESC
LIMIT 5;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


InvoiceId,Total
404,25.86
299,23.86
96,21.86
194,21.86
89,18.86


### Conditional Statements
Using a CASE statement to categorize invoice totals:

In [10]:
%%sql
SELECT InvoiceId,
       Total,
       CASE 
         WHEN Total < 10 THEN 'Low'
         WHEN Total BETWEEN 10 AND 20 THEN 'Medium'
         ELSE 'High'
       END AS TotalCategory
FROM Invoice
LIMIT 5;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


InvoiceId,Total,TotalCategory
1,1.98,Low
2,3.96,Low
3,5.94,Low
4,8.91,Low
5,13.86,Medium


## 4. Aggregate Functions & Group Summaries
Aggregate functions help summarize data.

### 4.1 Summary Statistics
Get the count and average of invoices:

In [11]:
%%sql
SELECT COUNT(*) AS InvoiceCount,
       AVG(Total) AS AverageTotal
FROM Invoice;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


InvoiceCount,AverageTotal
412,5.651941747572815


### 4.2 Group Summary
Show total sales per country:

In [12]:
%%sql
SELECT BillingCountry, COUNT(*) AS NumInvoices, SUM(Total) AS TotalSales
FROM Invoice
GROUP BY BillingCountry
ORDER BY TotalSales DESC;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


BillingCountry,NumInvoices,TotalSales
USA,91,523.06
Canada,56,303.96
France,35,195.1
Brazil,35,190.1
Germany,28,156.48
United Kingdom,21,112.86
Czech Republic,14,90.24
Portugal,14,77.24
India,13,75.26
Chile,7,46.62


### 4.3 Multiple Group Summary
Group by both country and year (using the InvoiceDate):

In [13]:
%%sql
SELECT BillingCountry,
       strftime('%Y', InvoiceDate) AS Year,
       COUNT(*) AS NumInvoices,
       SUM(Total) AS TotalSales
FROM Invoice
GROUP BY BillingCountry, Year
ORDER BY BillingCountry, Year;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


BillingCountry,Year,NumInvoices,TotalSales
Argentina,2010,3,11.88
Argentina,2011,1,0.99
Argentina,2013,3,24.75
Australia,2009,3,11.88
Australia,2010,1,0.99
Australia,2011,1,1.98
Australia,2012,2,22.77
Austria,2009,1,1.98
Austria,2010,2,27.77
Austria,2012,3,11.88


## 5 Combining Tables: Joins & Set Operators
### 5.1 Joins and Other Clauses
#### INNER JOIN Example
List invoices along with the customer’s first and last name:

In [14]:
%%sql
SELECT i.InvoiceId, i.Total, c.FirstName, c.LastName
FROM Invoice i
INNER JOIN Customer c ON i.CustomerId = c.CustomerId
LIMIT 5;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


InvoiceId,Total,FirstName,LastName
1,1.98,Leonie,Köhler
2,3.96,Bjørn,Hansen
3,5.94,Daan,Peeters
4,8.91,Mark,Philips
5,13.86,John,Gordon


#### LEFT JOIN Example
Show all customers, even if they have no invoices:

In [15]:
%%sql
SELECT c.CustomerId, c.FirstName, c.LastName, i.InvoiceId
FROM Customer c
LEFT JOIN Invoice i ON c.CustomerId = i.CustomerId
LIMIT 5;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


CustomerId,FirstName,LastName,InvoiceId
1,Luís,Gonçalves,98
1,Luís,Gonçalves,121
1,Luís,Gonçalves,143
1,Luís,Gonçalves,195
1,Luís,Gonçalves,316


### 5.2 Less Common Joins
#### Self Join Example
List pairs of artists with similar names (for demonstration):

In [16]:
%%sql
SELECT a1.ArtistId AS Artist1, a1.Name AS Name1,
       a2.ArtistId AS Artist2, a2.Name AS Name2
FROM Artist a1
JOIN Artist a2 ON a1.ArtistId < a2.ArtistId
WHERE a1.Name LIKE a2.Name || '%'
LIMIT 5;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


Artist1,Name1,Artist2,Name2
25,Milton Nascimento & Bebeto,42,Milton Nascimento
71,Vinícius De Moraes & Baden Powell,72,Vinícius De Moraes
122,R.E.M. Feat. Kate Pearson,124,R.E.M.
123,R.E.M. Feat. KRS-One,124,R.E.M.


### 5.3 Set Operators
#### UNION Example
Combine results from two similar queries (e.g., list distinct countries from Customer and Invoice tables):

In [17]:
%%sql
SELECT BillingCountry AS Country
FROM Invoice
UNION
SELECT Country
FROM Customer
ORDER BY Country;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


Country
Argentina
Australia
Austria
Belgium
Brazil
Canada
Chile
Czech Republic
Denmark
Finland


## 6 Querying SQL from Python
Using Python’s sqlite3 module along with pandas for data analysis:

In [18]:
# Example: Load total sales by country into a DataFrame
query = """
SELECT BillingCountry, SUM(Total) AS TotalSales
FROM Invoice
GROUP BY BillingCountry
ORDER BY TotalSales DESC;
"""
df_sales = pd.read_sql_query(query, conn)
df_sales.head()


,BillingCountry,TotalSales
0,USA,523.06
1,Canada,303.96
2,France,195.10
3,Brazil,190.10
4,Germany,156.48


You can also run SQL queries directly using the ipython-sql magic and capture the result in a variable:

In [19]:
result = %sql SELECT BillingCountry, SUM(Total) AS TotalSales FROM Invoice GROUP BY BillingCountry ORDER BY TotalSales DESC;
df_sales_magic = result.DataFrame()
df_sales_magic.head()

 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


,BillingCountry,TotalSales
0,USA,523.06
1,Canada,303.96
2,France,195.10
3,Brazil,190.10
4,Germany,156.48


## 7 SQL Subqueries
Subqueries allow you to nest one query inside another.

### 7.1 Scalar Subquery
Find customers whose total invoice amount is above the overall average:

In [20]:
%%sql
SELECT CustomerId, 
       (SELECT SUM(Total) FROM Invoice WHERE CustomerId = c.CustomerId) AS CustomerTotal
FROM Customer c
WHERE (SELECT AVG(Total) FROM Invoice) < (SELECT SUM(Total) FROM Invoice WHERE CustomerId = c.CustomerId)
LIMIT 5;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


CustomerId,CustomerTotal
1,39.62
3,39.62
12,37.62
15,38.62
18,37.62


### 7.2 Multi-row and Multi-column Subqueries
List invoices whose total is above the average total per country:

In [21]:
%%sql
SELECT InvoiceId, Total, BillingCountry
FROM Invoice
WHERE Total > (
    SELECT AVG(Total)
    FROM Invoice AS i2
    WHERE i2.BillingCountry = Invoice.BillingCountry
)
LIMIT 5;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


InvoiceId,Total,BillingCountry
3,5.94,Belgium
4,8.91,Canada
5,13.86,USA
11,8.91,United Kingdom
12,13.86,Germany


### 7.3 Nested and Correlated Subqueries
Find customers with more than one invoice:

In [22]:
%%sql
SELECT CustomerId, FirstName, LastName
FROM Customer c
WHERE 1 < (
    SELECT COUNT(*)
    FROM Invoice i
    WHERE i.CustomerId = c.CustomerId
);


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


CustomerId,FirstName,LastName
1,Luís,Gonçalves
2,Leonie,Köhler
3,François,Tremblay
4,Bjørn,Hansen
5,František,Wichterlová
6,Helena,Holý
7,Astrid,Gruber
8,Daan,Peeters
9,Kara,Nielsen
10,Eduardo,Martins


## 8 Common Table Expressions (CTE) & Views
### 8.1 Common Table Expression (CTE)
Calculate total sales per customer and then list the top 5:

In [23]:
%%sql
WITH CustomerSales AS (
    SELECT CustomerId, SUM(Total) AS TotalSpent
    FROM Invoice
    GROUP BY CustomerId
)
SELECT cs.CustomerId, cs.TotalSpent, c.FirstName, c.LastName
FROM CustomerSales cs
JOIN Customer c ON cs.CustomerId = c.CustomerId
ORDER BY cs.TotalSpent DESC
LIMIT 5;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


CustomerId,TotalSpent,FirstName,LastName
6,49.62,Helena,Holý
26,47.62,Richard,Cunningham
57,46.62,Luis,Rojas
45,45.62,Ladislav,Kovács
46,45.62,Hugh,O'Reilly


### 8.2 Creating and Querying a View
Create a view for high-value invoices (for demonstration):

In [30]:
%%sql
DROP VIEW IF EXISTS HighValueInvoices;

CREATE VIEW HighValueInvoices AS
SELECT InvoiceId, CustomerId, Total
FROM Invoice
WHERE Total > 20;



 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.
Done.


[]

Query the view:

In [31]:
%%sql
SELECT * FROM HighValueInvoices
LIMIT 5;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


InvoiceId,CustomerId,Total
96,45,21.86
194,46,21.86
299,26,23.86
404,6,25.86


In [32]:
query = "SELECT * FROM HighValueInvoices;"
df_high_value = pd.read_sql_query(query, conn)
df_high_value


,InvoiceId,CustomerId,Total
0,96,45,21.86
1,194,46,21.86
2,299,26,23.86
3,404,6,25.86


## 9. Window Functions
Window functions allow you to perform calculations across a set of rows related to the current row.

For example, calculate a running total of invoice amounts:

In [26]:
%%sql
SELECT InvoiceId, InvoiceDate, Total,
       SUM(Total) OVER (ORDER BY InvoiceDate) AS RunningTotal
FROM Invoice
ORDER BY InvoiceDate
LIMIT 10;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


InvoiceId,InvoiceDate,Total,RunningTotal
1,2009-01-01 00:00:00,1.98,1.98
2,2009-01-02 00:00:00,3.96,5.9399999999999995
3,2009-01-03 00:00:00,5.94,11.88
4,2009-01-06 00:00:00,8.91,20.79
5,2009-01-11 00:00:00,13.86,34.65
6,2009-01-19 00:00:00,0.99,35.64
7,2009-02-01 00:00:00,1.98,39.6
8,2009-02-01 00:00:00,1.98,39.6
9,2009-02-02 00:00:00,3.96,43.56
10,2009-02-03 00:00:00,5.94,49.5


Another example: rank customers by total spending

In [28]:
%%sql
WITH CustomerSpending AS (
    SELECT 
        c.CustomerId, 
        c.FirstName, 
        c.LastName, 
        SUM(i.Total) AS CustomerTotal
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    GROUP BY c.CustomerId
)
SELECT 
    CustomerId, 
    FirstName, 
    LastName, 
    CustomerTotal,
    RANK() OVER (ORDER BY CustomerTotal DESC) AS SpendingRank
FROM CustomerSpending
ORDER BY SpendingRank
LIMIT 5;


 * sqlite:///../data/Chinook_Sqlite.sqlite
Done.


CustomerId,FirstName,LastName,CustomerTotal,SpendingRank
6,Helena,Holý,49.62,1
26,Richard,Cunningham,47.62,2
57,Luis,Rojas,46.62,3
45,Ladislav,Kovács,45.62,4
46,Hugh,O'Reilly,45.62,4


## 10. Conclusion
In this notebook we have covered:

- The basics of SQL and how to explore a database schema.
- Writing SELECT statements with filtering, sorting, and conditional logic.
- Using aggregate functions and grouping data.
- Combining data from multiple tables using various types of joins and set operators.
- Advanced SQL topics including subqueries, common table expressions, views, and window functions.
- How to query SQL databases directly from Python using both sqlite3 and pandas.

This notebook provides a comprehensive introduction to SQL that you can extend with your own projects and more advanced queries. Enjoy exploring your data with SQL!